# Ramen 修正版 v4：独立训练与可追溯评估

适用于云盘已有 Ramen 数据的实验。选择 L4；默认只运行新的 v4 联合模型和顺序语义基线，上限 15000 轮、RGB 预热 1500 轮，保留 12 个验证视角。旧 v3 权重不覆盖。

训练与可下载的预训练缓存位于 Colab 临时盘，检查点会备份到 Drive。建议额外预留至少 8 GiB；若同时补跑旧 v3 基线，建议至少 13 GiB。这不是峰值保证：高斯数量及 Drive 回收站会增加占用。不要在云盘满额时启动。脚本不会永久清空回收站。

v4 修复投影坐标、语义编码的验证集隔离、验证全通道评估和实际支持的 SH3。代码修正不等于效果提升，必须等待两个模型完成后查看报告。断线恢复若发现语义空间不兼容会停止，不能把新编码与旧权重混用。

In [ ]:
import subprocess, sys
from pathlib import Path
import torch
assert torch.cuda.is_available(), '请选择 GPU 运行时'
print('GPU:', torch.cuda.get_device_name(0))
REPO = Path('/content/gaussian-splatting')
if (REPO/'.git').is_dir():
    subprocess.run(['git','-C',str(REPO),'pull','--ff-only'], check=True)
else:
    subprocess.run(['git','clone','--recursive','https://github.com/Xuyw041006-arch/gaussian-splatting.git',str(REPO)], check=True)
subprocess.run(['git','submodule','update','--init','--recursive'], cwd=REPO, check=True)
subprocess.run([sys.executable,'-m','pip','install','-q','plyfile','open-clip-torch','scikit-learn','ftfy','regex','opencv-python-headless','git+https://github.com/facebookresearch/segment-anything.git'], check=True)
subprocess.run([sys.executable,'-m','pip','install','-q','./submodules/diff-gaussian-rasterization','./submodules/simple-knn','./submodules/fused-ssim'], cwd=REPO, check=True)
result = subprocess.run([sys.executable,'-m','unittest','discover','-s','tests','-p','test_*.py','-v'], cwd=REPO, capture_output=True, text=True)
print(result.stdout, result.stderr)
result.check_returncode()

In [ ]:
from google.colab import drive
import shutil, urllib.request, hashlib
drive.mount('/content/drive')
PERSIST_ROOT = Path('/content/drive/MyDrive/semantic_adaptive_3dgs')
WORK_ROOT = Path('/content/ramen_recovery')
RECOVER_OLD_V3 = False  # True 还会补跑旧 v3 顺序基线；v4 对照始终运行。
MIN_DRIVE_GIB = 13 if RECOVER_OLD_V3 else 8
assert (PERSIST_ROOT/'ramen_detail_v2_15k/data/ramen').is_dir(), '缺少已有 Ramen 数据集'
free = shutil.disk_usage(PERSIST_ROOT).free / 2**30
print('Drive 挂载层报告可用 GiB:', round(free, 2), '；同时检查账户与回收站配额')
assert free >= MIN_DRIVE_GIB, f'请先释放云盘空间，至少预留 {MIN_DRIVE_GIB} GiB'
SAM = Path('/content/ramen_recoverable_cache/ramen_detail_v2_15k/assets/sam_vit_h_4b8939.pth')
if not SAM.is_file():
    SAM = Path('/content/ramen_assets/sam_vit_h_4b8939.pth')
    SAM.parent.mkdir(parents=True, exist_ok=True)
    if not SAM.is_file():
        partial = SAM.with_suffix('.download')
        urllib.request.urlretrieve('https://dl.fbaipublicfiles.com/segment_anything/sam_vit_h_4b8939.pth', partial)
        partial.replace(SAM)
digest = hashlib.sha256()
with SAM.open('rb') as handle:
    for block in iter(lambda: handle.read(8*1024*1024), b''): digest.update(block)
assert digest.hexdigest() == 'a7bf3b02f3ebf1267aba913ff637d9a2d5c33d3173bb679e46d9f338c26f262e', 'SAM 缓存校验失败，不开始训练'
print('SAM 缓存校验完成:', SAM)

In [ ]:
command = [sys.executable,'-u','-m','scripts.run_ramen_recovery','--persist_root',str(PERSIST_ROOT),'--work_root',str(WORK_ROOT),'--sam_checkpoint',str(SAM),'--min_drive_free_gb',str(MIN_DRIVE_GIB)]
if not RECOVER_OLD_V3: command.append('--skip_v3_recovery')
print('阶段状态：', WORK_ROOT/'recovery_status.json')
print('详细训练日志：', WORK_ROOT/'logs')
with subprocess.Popen(command, cwd=REPO, stdout=subprocess.PIPE, stderr=subprocess.STDOUT, text=True, bufsize=1) as process:
    for line in process.stdout: print(line, end='', flush=True)
    if process.wait(): raise RuntimeError('流程未完成，请查看 recovery_status.json 和对应日志；不要清空权重')

In [ ]:
from IPython.display import Markdown, display
report = WORK_ROOT/'ramen_corrected_v4_15k/outputs_full/report/ramen_final_report.md'
assert report.is_file(), '报告尚未生成，不能视为训练完成'
display(Markdown(report.read_text()))